# Statistik mit R

Hypothesentests, Regression und Konfidenzintervalle unter Verwendung integrierter R-Datensätze.

Kein Datendownload oder Paketinstallation erforderlich — verwendet ausschließlich Basis-R.

## 1. Deskriptive Statistik

In [ ]:
data(mtcars)
cat("Dataset: mtcars (", nrow(mtcars), "cars, ", ncol(mtcars), "variables)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. Zweistichproben-t-Test

Haben Autos mit manuellem Schaltgetriebe einen besseren Kraftstoffverbrauch (MPG) als Automatikfahrzeuge?

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("Automatic:", round(mean(auto), 1), "mpg (n =", length(auto), ")\n")
cat("Manual:   ", round(mean(manual), 1), "mpg (n =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nConclusion:",
    ifelse(t_result$p.value < 0.05,
           "Reject H0 — manual cars have significantly higher MPG",
           "Fail to reject H0"))

## 3. Chi-Quadrat-Test

Sind Zylinderzahl und Getriebetyp unabhängig voneinander?

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("Automatic", "Manual")
print(tab)
cat("\n")
chisq.test(tab)

## 4. Multiple lineare Regression

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. Regressionsdiagnostik

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. Konfidenzintervalle

In [ ]:
ci <- confint(model, level = 0.95)
cat("95% Confidence Intervals:\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "Estimate", ylab = "",
     main = "95% Confidence Intervals for Coefficients")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. Einfaktorielle Varianzanalyse (One-Way ANOVA)

Unterscheidet sich der Kraftstoffverbrauch (MPG) signifikant zwischen den Zylinderzahlen?

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nTukey HSD Post-hoc Comparisons:\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "MPG by Cylinder Count",
        xlab = "Cylinders", ylab = "Miles per Gallon",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## Zusammenfassung

- **Welch-t-Test**: Im unbereinigten einseitigen Vergleich weisen Autos mit Schaltgetriebe einen höheren durchschnittlichen MPG-Wert auf
- **Chi-Quadrat**: Die Kontingenztabelle deutet auf einen Zusammenhang hin, aber die geringen erwarteten Häufigkeiten lösen eine Approximationswarnung aus; interpretieren Sie dieses Ergebnis daher mit Vorsicht
- **Regression**: Gewicht und PS-Zahl (Horsepower) sind nach der Bereinigung signifikante negative Prädiktoren; der Getriebetyp ist in diesem Modell nicht signifikant
- **ANOVA**: Der MPG-Wert unterscheidet sich signifikant zwischen den 4-, 6- und 8-Zylinder-Gruppen; die Tukey-Ergebnisse identifizieren die paarweisen Unterschiede